# Inter-patient evaluation (DS1 / DS2 split)

**Why this notebook exists.** The main notebook uses the Kaggle `mitbih_train.csv` / `mitbih_test.csv` files.
Those files split the data **by beat**, so beats of the same patient are in both train and test.
That probably makes the scores too optimistic.

This notebook repeats the comparison with a **patient-wise split**, the DS1 / DS2 protocol of de Chazal et al. (2004):
22 recordings for training (DS1) and 22 different recordings for testing (DS2). No patient is on both sides.

**What is the same:** the three models, the augmentation, the class weights and the training settings.

**What is different:**
- Beats are cut from the raw PhysioNet records with the cardiologist R-peak annotations. The Kaggle files used the
  method of Kachuee et al. (2018), which detects peaks itself. The windows are built the same way (start at the R-peak,
  1.2 x median R-R long, 125 Hz, scaled to 0..1, zero-padded to 187), but they are not identical.
- The 4 paced recordings (102, 104, 107, 217) are excluded, as the protocol says. Almost all "Unknown" (Q) beats are
  paced beats, so the Q class nearly disappears. This notebook therefore uses **4 classes: N, S, V, F**.

**Before running on Kaggle:** turn the GPU on and turn **Internet on** (the records are downloaded from PhysioNet).

In [ ]:
!pip -q install wfdb

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
import wfdb
from scipy.signal import resample_poly
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Conv1D, BatchNormalization, MaxPooling1D,
    Dense, Dropout, SpatialDropout1D, Flatten, Concatenate,
    GlobalAveragePooling1D, Add, MultiHeadAttention, LayerNormalization
)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam
from sklearn.utils import shuffle
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

SEED = 42            # change this to repeat the run with another initialisation
tf.random.set_seed(SEED)
np.random.seed(SEED)

CLASS_NAMES = ['Normal', 'Supraventricular', 'Ventricular', 'Fusion']
NUM_CLASSES = 4
BEAT_LEN    = 187    # samples per beat at 125 Hz, same as the Kaggle files
TARGET_FS   = 125

# quick settings (leave as they are for the real run)
EPOCHS      = int(os.environ.get('EPOCHS', 50))
DATA_DIR    = os.environ.get('MITDB_DIR', 'mitdb')

print(f'TensorFlow {tf.__version__}')
print(f'GPU: {tf.config.list_physical_devices("GPU")}')

## 1. Patient-wise split and beat extraction

In [ ]:
# DS1 / DS2 record lists from de Chazal et al. (2004). 22 recordings each.
# The paced recordings 102, 104, 107 and 217 are in neither list.
DS1 = [101, 106, 108, 109, 112, 114, 115, 116, 118, 119, 122, 124,
       201, 203, 205, 207, 208, 209, 215, 220, 223, 230]
DS2 = [100, 103, 105, 111, 113, 117, 121, 123, 200, 202, 210, 212,
       213, 214, 219, 221, 222, 228, 231, 232, 233, 234]
assert len(DS1) == 22 and len(DS2) == 22 and not set(DS1) & set(DS2)

# MIT-BIH beat symbols -> AAMI class index (N=0, S=1, V=2, F=3). Q beats are skipped.
AAMI = {}
for sym in ['N', 'L', 'R', 'e', 'j']:
    AAMI[sym] = 0
for sym in ['A', 'a', 'J', 'S']:
    AAMI[sym] = 1
for sym in ['V', 'E']:
    AAMI[sym] = 2
AAMI['F'] = 3
Q_SYMBOLS = ['/', 'f', 'Q']


def extract_beats(record_path):
    """Cut one record into beats. Returns (X, y, n_skipped_q).

    X has shape (n_beats, 187): each beat starts at its annotated R-peak, is
    1.2 x the median R-R interval long, is scaled to 0..1 and zero-padded.
    """
    record = wfdb.rdrecord(record_path, channel_names=['MLII'])
    ann = wfdb.rdann(record_path, 'atr')
    fs = int(record.fs)

    signal = resample_poly(record.p_signal[:, 0], TARGET_FS, fs)       # 360 Hz -> 125 Hz
    peaks = np.round(ann.sample * TARGET_FS / fs).astype(int)
    symbols = np.array(ann.symbol)

    is_beat = np.isin(symbols, list(AAMI) + Q_SYMBOLS)
    beat_peaks, beat_symbols = peaks[is_beat], symbols[is_beat]
    median_rr = np.median(np.diff(beat_peaks))
    window = int(min(BEAT_LEN, round(1.2 * median_rr)))

    X, y, skipped_q = [], [], 0
    for peak, sym in zip(beat_peaks, beat_symbols):
        if sym in Q_SYMBOLS:
            skipped_q += 1
            continue
        segment = signal[peak:peak + window]
        if len(segment) < window:
            continue                                   # beat cut off by the end of the record
        low, high = segment.min(), segment.max()
        if high - low < 1e-6:
            continue                                   # flat line, nothing to learn
        beat = np.zeros(BEAT_LEN, dtype=np.float32)
        beat[:window] = (segment - low) / (high - low)
        X.append(beat)
        y.append(AAMI[sym])
    return np.array(X, dtype=np.float32).reshape(-1, BEAT_LEN), np.array(y, dtype=int), skipped_q


def load_split(records, data_dir):
    X_all, y_all, groups, skipped = [], [], [], 0
    for rec in records:
        X, y, q = extract_beats(os.path.join(data_dir, str(rec)))
        X_all.append(X); y_all.append(y); groups.append(np.full(len(y), rec)); skipped += q
    return np.vstack(X_all), np.concatenate(y_all), np.concatenate(groups), skipped

In [ ]:
if not os.path.exists(os.path.join(DATA_DIR, '100.hea')):
    print('Downloading the MIT-BIH Arrhythmia Database from PhysioNet...')
    wfdb.dl_database('mitdb', dl_dir=DATA_DIR)

X_train_raw, y_train_raw, train_records, q_train = load_split(DS1, DATA_DIR)
X_test_raw,  y_test,      test_records,  q_test  = load_split(DS2, DATA_DIR)

# the whole point of this notebook: no recording is on both sides
assert not set(train_records) & set(test_records)

print(f'Train (DS1): {X_train_raw.shape} | Test (DS2): {X_test_raw.shape}')
print(f'Q beats skipped: {q_train} in DS1, {q_test} in DS2')
print(f"\n{'Class':20s} {'DS1 train':>10s} {'DS2 test':>10s}")
for i, name in enumerate(CLASS_NAMES):
    print(f'{name:20s} {np.sum(y_train_raw == i):10d} {np.sum(y_test == i):10d}')

## 2. Augmentation and preprocessing (same as the main notebook)

In [ ]:
def augment_beat(signal, shift_range=10, scale_range=0.12, noise_std=0.01):
    """
    Apply three ECG-valid augmentations to a single 187-timestep beat.
    
    Args:
        signal      : 1D array of length 187
        shift_range : max time-shift in timesteps (±10 = ±80ms at 125Hz)
        scale_range : amplitude scaling ± this fraction
        noise_std   : standard deviation of additive Gaussian noise
    Returns:
        Augmented signal of same length 187
    """
    # 1. Random time-shift — roll the signal and zero-pad the wrapped edge
    shift = np.random.randint(-shift_range, shift_range)
    aug   = np.roll(signal, shift)
    if shift > 0:
        aug[:shift] = 0.0
    elif shift < 0:
        aug[shift:] = 0.0

    # 2. Random amplitude scaling (multiplicative, mean=1.0)
    scale = 1.0 + np.random.uniform(-scale_range, scale_range)
    aug   = aug * scale

    # 3. Additive Gaussian noise — simulates electrode noise
    aug = aug + np.random.normal(0, noise_std, aug.shape)

    # Clip to valid range [0, 1] (MIT-BIH signals are pre-normalised)
    return np.clip(aug, 0.0, 1.0)


def augment_minority_classes(X, y, target_count=5000, aug_classes=(1, 2, 3)):
    """
    Over-sample minority classes up to target_count by augmentation.
    
    Args:
        X             : raw signals, shape (N, 187)
        y             : integer labels, shape (N,)
        target_count  : desired minimum samples per minority class
        aug_classes   : which class indices to augment (S=1, V=2, F=3)
    Returns:
        X_aug, y_aug  : augmented dataset (original + synthetic samples)
    """
    X_extra, y_extra = [], []
    for cls in aug_classes:
        mask      = (y == cls)
        X_cls     = X[mask]           # all real samples for this class
        n_real    = len(X_cls)
        n_needed  = max(0, target_count - n_real)
        if n_needed == 0:
            continue
        # Randomly pick which real samples to augment (with replacement)
        idxs = np.random.choice(n_real, n_needed, replace=True)
        for idx in idxs:
            X_extra.append(augment_beat(X_cls[idx]))
            y_extra.append(cls)
        print(f'  {CLASS_NAMES[cls]:20s}: {n_real:5d} → {n_real + n_needed} (+{n_needed} synthetic)')

    if X_extra:
        X_aug = np.vstack([X, np.array(X_extra, dtype=np.float32)])
        y_aug = np.concatenate([y, np.array(y_extra, dtype=int)])
    else:
        X_aug, y_aug = X, y
    return X_aug, y_aug


print('=== AUGMENTATION ===')
X_train_aug, y_train_aug = augment_minority_classes(
    X_train_raw, y_train_raw, target_count=5000
)
print(f'\nDataset size after augmentation: {len(X_train_aug)}')

In [ ]:
X_train_r = X_train_aug.reshape(-1, BEAT_LEN, 1)
X_test    = X_test_raw.reshape(-1, BEAT_LEN, 1)

weights       = compute_class_weight('balanced', classes=np.arange(NUM_CLASSES), y=y_train_aug)
class_weights = dict(enumerate(weights))
for i, name in enumerate(CLASS_NAMES):
    print(f'  {name:20s}: weight {class_weights[i]:.4f}')

y_train_cat = tf.keras.utils.to_categorical(y_train_aug, num_classes=NUM_CLASSES)
X_train, y_train = shuffle(X_train_r, y_train_cat, random_state=SEED)
print(f'X_train: {X_train.shape} | X_test: {X_test.shape}')

## 3. Models (copied unchanged from the main notebook)

In [ ]:
# ──────────────────────────────────────────────────────────────
# SHARED BUILDING BLOCKS & EVALUATION UTILITIES
# ──────────────────────────────────────────────────────────────

def dense_block(x, filters):
    """
    DenseNet block: each layer receives all previous feature maps.
    Output = concatenation of [input, conv1_out, conv2_out].
    This dense connectivity provides two benefits:
      - Feature reuse: early low-level ECG features survive to later layers
      - Implicit deep supervision: gradients flow directly to early layers
    """
    c1 = Conv1D(filters, 3, padding='same', activation='relu')(x)
    c1 = BatchNormalization()(c1)
    c1 = SpatialDropout1D(0.1)(c1)   # Drops entire feature maps, more effective than Dropout
    m1 = Concatenate()([x, c1])

    c2 = Conv1D(filters, 3, padding='same', activation='relu')(m1)
    c2 = BatchNormalization()(c2)
    c2 = SpatialDropout1D(0.1)(c2)
    return Concatenate()([x, c1, c2])


def transition_layer(x, out_filters):
    """
    Transition layer between dense blocks.
    Problem it solves: dense_block output has x + c1 + c2 channels.
    After 3 blocks (32→64→128), you'd have ~500 channels WITHOUT this.
    The 1×1 conv compresses back to out_filters (halving), keeping
    the model lean and reducing overfitting risk.
    This is the standard DenseNet-BC design (Huang et al., 2017).
    """
    x = Conv1D(out_filters, 1, padding='same', activation='relu')(x)  # 1×1 compression
    x = BatchNormalization()(x)
    return MaxPooling1D(2)(x)  # Halve temporal resolution


def attention_block(x, num_heads=4, key_dim=16, name_prefix='attn'):
    """
    Multi-Head Self-Attention block with residual connection.
    
    How it helps ECG classification:
    - Attention heads learn to focus on different ECG regions simultaneously
      (one head for QRS complex, another for T-wave, etc.)
    - The residual Add() preserves original features so the block
      is additive — it can only enhance, never destroy, existing features
    - LayerNorm stabilises training by normalising across feature dimension
    """
    # Project to attention dimension first (reduces computation)
    proj = Conv1D(num_heads * key_dim, 1, padding='same', activation='relu',
                  name=f'{name_prefix}_proj')(x)

    # Self-attention: Q=K=V=proj (the signal attends to itself)
    attn = MultiHeadAttention(
        num_heads=num_heads, key_dim=key_dim,
        dropout=0.1, name=f'{name_prefix}_mha'
    )(proj, proj)

    # Project back to original channel size for the residual addition
    attn = Conv1D(x.shape[-1], 1, padding='same', name=f'{name_prefix}_back')(attn)

    # Residual connection + LayerNormalization
    x = Add(name=f'{name_prefix}_add')([x, attn])
    return LayerNormalization(name=f'{name_prefix}_ln')(x)

In [ ]:
def build_cnn():
    inp = Input(shape=(187, 1), name='ecg_input')

    # Block 1: broad filters (size 5) to capture QRS and early repolarisation
    x = Conv1D(32, 5, activation='relu', padding='same')(inp)
    x = BatchNormalization()(x)
    x = SpatialDropout1D(0.1)(x)   # Drop 10% of feature maps per step
    x = MaxPooling1D(2)(x)          # 187 → 93 timesteps

    # Block 2: medium filters to capture QRS complex shape
    x = Conv1D(64, 5, activation='relu', padding='same')(x)
    x = BatchNormalization()(x)
    x = SpatialDropout1D(0.1)(x)
    x = MaxPooling1D(2)(x)          # 93 → 46 timesteps

    # Block 3: narrow filters (size 3) for fine T-wave morphology
    x = Conv1D(128, 3, activation='relu', padding='same')(x)
    x = BatchNormalization()(x)
    x = SpatialDropout1D(0.1)(x)
    x = MaxPooling1D(2)(x)          # 46 → 23 timesteps

    x = Flatten()(x)
    x = Dense(64, activation='relu')(x)
    x = Dropout(0.3)(x)             # Standard dropout in dense head
    out = Dense(NUM_CLASSES, activation='softmax')(x)

    return Model(inp, out, name='CNN_Baseline')

In [ ]:
def build_densenet_improved():
    inp = Input(shape=(187, 1), name='ecg_input')

    # ── Stage 1 ────────────────────────────────────────────────
    # Initial conv to establish 32-channel feature space
    x = Conv1D(32, 5, padding='same', activation='relu')(inp)
    x = BatchNormalization()(x)
    # Dense block: each conv sees ALL previous outputs
    x = dense_block(x, 32)
    # Transition: compress back to 32 channels and halve time resolution
    # Without this, the concatenated output would be ~96 channels here
    x = transition_layer(x, out_filters=32)

    # ── Stage 2 ────────────────────────────────────────────────
    x = Conv1D(64, 3, padding='same', activation='relu')(x)
    x = BatchNormalization()(x)
    x = dense_block(x, 64)
    x = transition_layer(x, out_filters=64)  # Compresses ~192 → 64

    # ── Stage 3 ────────────────────────────────────────────────
    x = Conv1D(128, 3, padding='same', activation='relu')(x)
    x = BatchNormalization()(x)
    x = dense_block(x, 128)
    x = transition_layer(x, out_filters=128)  # Compresses ~384 → 128

    # Global average pooling collapses the temporal dimension
    # to a single 128-dimensional feature vector per beat
    x = GlobalAveragePooling1D()(x)

    # Dense classification head with L2 regularisation + dropout
    x = Dense(128, activation='relu',
              kernel_regularizer=tf.keras.regularizers.l2(1e-4))(x)
    x = Dropout(0.4)(x)
    x = Dense(64, activation='relu',
              kernel_regularizer=tf.keras.regularizers.l2(1e-4))(x)
    x = Dropout(0.3)(x)
    out = Dense(NUM_CLASSES, activation='softmax')(x)

    return Model(inp, out, name='DenseNet_Improved')

In [ ]:
def build_proposed_model():
    inp = Input(shape=(187, 1), name='ecg_input')

    # ── Stage 1: Low-level feature extraction ─────────────────
    # Broad kernel (5) captures slow waveform components (ST segment and T-wave)
    x = Conv1D(32, 5, padding='same', activation='relu')(inp)
    x = BatchNormalization()(x)
    x = dense_block(x, 32)
    x = transition_layer(x, out_filters=32)   # 187 → 93 timesteps

    # ── Stage 2: Mid-level feature extraction ─────────────────
    x = Conv1D(64, 3, padding='same', activation='relu')(x)
    x = BatchNormalization()(x)
    x = dense_block(x, 64)
    x = transition_layer(x, out_filters=64)   # 93 → 46 timesteps

    # ── ATTENTION BLOCK 1: Mid-scale (46 timesteps ≈ 128ms) ───
    # At this resolution, the model can learn which temporal segments
    # (PR interval, QRS onset, ST segment) are most discriminative
    # for each arrhythmia class.
    # 4 heads × key_dim 16 = 64 attention dimensions
    x = attention_block(x, num_heads=4, key_dim=16, name_prefix='attn1')

    # ── Stage 3: High-level feature extraction ────────────────
    x = Conv1D(128, 3, padding='same', activation='relu')(x)
    x = BatchNormalization()(x)
    x = dense_block(x, 128)
    x = transition_layer(x, out_filters=128)  # 46 → 23 timesteps

    # ── ATTENTION BLOCK 2: Fine-scale (23 timesteps ≈ 64ms) ───
    # At this compressed resolution, attention learns which of the
    # high-level features (e.g., QRS notch, ST elevation) are
    # diagnostically critical for the final classification.
    # 8 heads for richer attention diversity at the decision layer.
    x = attention_block(x, num_heads=8, key_dim=16, name_prefix='attn2')

    # Global average pooling → 128-dim beat representation
    x = GlobalAveragePooling1D()(x)

    # ── Classification head ───────────────────────────────────
    x = Dense(128, activation='relu',
              kernel_regularizer=tf.keras.regularizers.l2(1e-4))(x)
    x = Dropout(0.4)(x)
    x = Dense(64, activation='relu',
              kernel_regularizer=tf.keras.regularizers.l2(1e-4))(x)
    x = Dropout(0.3)(x)
    out = Dense(NUM_CLASSES, activation='softmax', name='predictions')(x)

    return Model(inp, out, name='DenseNet_MultiScale_Attention')

## 4. Train and evaluate on DS2

Same settings as the main notebook: Adam, categorical cross-entropy, batch size 64, up to 50 epochs,
early stopping on validation accuracy (patience 7), learning rate halved on a plateau.
The validation part is 15% of the DS1 beats.

In [ ]:
def train_and_evaluate(name, build_fn, learning_rate, clipnorm=None):
    tf.keras.backend.clear_session()
    tf.random.set_seed(SEED)
    model = build_fn()
    optimizer = Adam(learning_rate=learning_rate, clipnorm=clipnorm) if clipnorm else Adam(learning_rate=learning_rate)
    model.compile(optimizer=optimizer, loss='categorical_crossentropy', metrics=['accuracy'])
    model.fit(
        X_train, y_train,
        epochs=EPOCHS, batch_size=64, validation_split=0.15,
        class_weight=class_weights, verbose=2,
        callbacks=[
            EarlyStopping(monitor='val_accuracy', patience=7, restore_best_weights=True, verbose=1),
            ReduceLROnPlateau(monitor='val_accuracy', factor=0.5, patience=3, min_lr=1e-7, verbose=1),
        ],
    )
    y_pred = np.argmax(model.predict(X_test, verbose=0), axis=1)
    labels = list(range(NUM_CLASSES))
    print(f'\n=== {name} on DS2 (unseen patients) ===')
    print(classification_report(y_test, y_pred, labels=labels, target_names=CLASS_NAMES, digits=4, zero_division=0))

    cm = confusion_matrix(y_test, y_pred, labels=labels)
    plt.figure(figsize=(7, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
    plt.title(f'{name}: DS2 confusion matrix'); plt.ylabel('True'); plt.xlabel('Predicted')
    plt.tight_layout(); plt.show()

    report = classification_report(y_test, y_pred, labels=labels, target_names=CLASS_NAMES,
                                   output_dict=True, zero_division=0)
    row = {'Model': name, 'Parameters': model.count_params(),
           'Accuracy': report['accuracy'], 'Macro F1': report['macro avg']['f1-score']}
    for cls in CLASS_NAMES[1:]:
        row[f'{cls} sensitivity'] = report[cls]['recall']
        row[f'{cls} PPV'] = report[cls]['precision']
    return row


results = []

In [ ]:
results.append(train_and_evaluate('CNN baseline', build_cnn, 0.001))

In [ ]:
results.append(train_and_evaluate('DenseNet', build_densenet_improved, 0.0005, clipnorm=1.0))

In [ ]:
results.append(train_and_evaluate('DenseNet + attention', build_proposed_model, 0.0005, clipnorm=1.0))

## 5. Results: patient-wise split against the by-beat split

The by-beat numbers are run 2 of the main notebook (5 classes, Kaggle files). The two rows are **not a like-for-like
comparison**: the class set and the beat extraction differ. Read the gap as an indication of how much the by-beat
split helped, not as an exact measurement.

In [ ]:
table = pd.DataFrame(results).set_index('Model').round(4)
table.to_csv('inter_patient_results.csv')
display(table)

by_beat = pd.DataFrame({
    'Model': ['CNN baseline', 'DenseNet', 'DenseNet + attention'],
    'By-beat accuracy': [0.9793, 0.9834, 0.9846],
    'By-beat macro F1': [0.903, 0.911, 0.909],
}).set_index('Model')
compare = by_beat.join(table[['Accuracy', 'Macro F1']].rename(
    columns={'Accuracy': 'Patient-wise accuracy', 'Macro F1': 'Patient-wise macro F1'}))
display(compare)